# respirAI: train the lung-sound model on ICBHI 2017

This notebook does tasks **D1–D6** from `docs/PLAN.md`, top to bottom:

1. **Explore** the ICBHI 2017 dataset (D1)
2. **Split by patient**, never randomly (D3)
3. **Preprocess** every recording with the shared `ai/preprocess.py` (D2)
4. **Train** a ResNet18 on log-mel spectrograms, 4 classes: normal / crackles / wheezes / both (D4)
5. **Test** honestly, with per-class scores, on patients the model has never seen
6. **Export** the model to ONNX for the server (D5) and pick the **demo clips** (D6)

## Before you press Run All

1. **Add the data:** in the right panel, *Input* → *+ Add Input* → search **Respiratory Sound Database** (by *vbookshelf*) → *Add*.
2. **Turn on a GPU:** *Settings* → *Accelerator* → **GPU T4 x2** or **GPU P100**.
3. **Turn on Internet:** *Settings* → *Internet* → **On**. Kaggle asks you to verify a phone number once. The notebook uses the internet to download the repo code and the ImageNet starting weights.
4. *Run All*. Or use *Save Version* → *Save & Run All* to run it in the background and close the tab.

The whole notebook takes roughly 10–20 minutes on a GPU. When it finishes, download **`respirai_export.zip`** from the *Output* section. The last cell explains where each file goes.

## 0. Setup

In [ ]:
# The code lives in the repo, so training and the server use exactly the same preprocessing.
REPO_URL = "https://github.com/denis-furtuna/respirAI.git"
BRANCH = "ai-training"  # switch to "main" once this is merged

import os, subprocess, sys
from pathlib import Path

CODE = Path("/tmp/respirAI")
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=False)
sys.path.insert(0, str(CODE / "ai"))
os.environ["PYTHONPATH"] = str(CODE / "ai")  # so parallel worker processes find it too

# onnx/onnxruntime: to export the model and test it exactly as the server will run it.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
print("code from", CODE / "ai")

In [ ]:
import json, random, shutil, time
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from joblib import Parallel, delayed
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset

import icbhi
import preprocess as pp
from icbhi import CLASSES, POINTS
from model import build_model, export

SEED = 42
EPOCHS = 30
BATCH = 64
LR = 1e-3
MODEL_VERSION = f"resp-cnn-{date.today():%Y%m%d}"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPORT = Path("/kaggle/working/export"); EXPORT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: Settings -> Accelerator)")

## 1. Explore the data (D1)

One row per recording. The file name tells us the patient and the chest point. The `.txt` next to each file lists the breathing cycles and whether each one has crackles or wheezes.

In [ ]:
AUDIO_DIR = icbhi.find_audio_dir()
df = icbhi.load_index(AUDIO_DIR)
print(f"{len(df)} recordings from {df.patient.nunique()} patients, {df.n_cycles.sum()} breathing cycles")
print(f"cycles with crackles: {df.crackle_cycles.sum()}, with wheezes: {df.wheeze_cycles.sum()}")
df.head()

In [ ]:
diag = icbhi.load_diagnosis()
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
df.point.value_counts().reindex(POINTS).plot.bar(ax=ax[0], title="recordings per chest point")
df.label.map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES).plot.bar(ax=ax[1], title="recordings by sound (any cycle)")
if diag is not None:
    diag.diagnosis.value_counts().plot.bar(ax=ax[2], title="patients by diagnosis (NOT used for training)")
plt.tight_layout(); plt.show()
print("Recording devices:", df.device.value_counts().to_dict())

## 2. Split by patient (D3)

A random split would put recordings of the same person in both train and test. The model would then recognise the person instead of the sound, and the accuracy would look far better than it really is. Every patient goes to exactly one split.

In [ ]:
split = icbhi.patient_split(df, seed=SEED)
df["split"] = df.patient.map(split)

sets = {s: set(df[df.split == s].patient) for s in ["train", "val", "test"]}
assert not (sets["train"] & sets["test"]) and not (sets["train"] & sets["val"]) and not (sets["val"] & sets["test"])
print({s: f"{len(p)} patients" for s, p in sets.items()})
df[["stem", "patient", "point", "split"]].to_csv(EXPORT / "split.csv", index=False)
pd.crosstab(df.split, df.label.map(dict(enumerate(CLASSES))))

## 3. Preprocess (D2)

Each recording is first made to look like what our device sends: 8 kHz, 16-bit. It then goes through the shared `preprocess.py`: 4 kHz, band-pass 100–1800 Hz, and 4-second windows (2 s apart) turned into log-mel spectrograms.

We train on **windows, not single cycles**, because the server will see a 15 s recording with no cycle markings. Each window gets the label of the cycles inside it.

In [ ]:
def process(row):
    audio, sr = sf.read(row.path, dtype="float32", always_2d=True)
    device_audio = icbhi.as_device(audio.mean(axis=1), sr)          # what the phone would upload
    wins, starts = pp.windows(pp.clean(device_audio, pp.DEVICE_SR))
    cycles = icbhi.read_cycles(row.path.replace(".wav", ".txt"))
    out = []
    for w, s in zip(wins, starts):
        label = icbhi.window_label(cycles, s)
        if label is not None:                                        # skip stretches with no marked cycle
            out.append((pp.log_mel(w).astype(np.float16), label, row.stem, s))
    return out

t0 = time.time()
results = Parallel(n_jobs=-1)(delayed(process)(r) for r in df.itertuples())
items = [it for rec in results for it in rec]
X = np.stack([it[0] for it in items])
y = np.array([it[1] for it in items])
win = pd.DataFrame({"stem": [it[2] for it in items], "start": [it[3] for it in items], "label": y})
win["split"] = win.stem.map(df.set_index("stem").split)
print(f"{len(X)} windows of shape {X.shape[1:]} in {time.time() - t0:.0f} s")
pd.crosstab(win.split, win.label.map(dict(enumerate(CLASSES))))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.2))
for c, name in enumerate(CLASSES):
    i = np.flatnonzero(y == c)[0]
    ax[c].imshow(X[i].astype(np.float32), origin="lower", aspect="auto", cmap="magma")
    ax[c].set_title(f"{name}  ({win.stem[i]})"); ax[c].set_xlabel("time frames (16 ms)"); ax[c].set_ylabel("mel band")
plt.tight_layout(); plt.show()

## 4. Train (D4)

ResNet18, starting from ImageNet weights, with one input channel and 4 outputs. "Normal" windows far outnumber the rest, so the loss gives rarer classes more weight. Light SpecAugment (masking random time and frequency stripes) helps the model generalise to new patients.

The model is picked by the **ICBHI score** on the validation patients: the average of specificity (normal recognised as normal) and sensitivity (abnormal sounds classified correctly). This is the standard metric for this dataset.

In [ ]:
class Windows(Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        x = X[i].astype(np.float32).copy()
        if self.train:
            x = np.roll(x, np.random.randint(x.shape[1]), axis=1)
            for _ in range(2):
                f = np.random.randint(0, 8); f0 = np.random.randint(0, x.shape[0] - f)
                x[f0:f0 + f] = 0
                t = np.random.randint(0, 30); t0 = np.random.randint(0, x.shape[1] - t)
                x[:, t0:t0 + t] = 0
        return torch.from_numpy(x)[None], int(y[i])

def icbhi_score(true, pred):
    true, pred = np.asarray(true), np.asarray(pred)
    sp = np.mean(pred[true == 0] == 0)
    se = np.mean(pred[true != 0] == true[true != 0])
    return {"specificity": float(sp), "sensitivity": float(se), "icbhi_score": float((sp + se) / 2)}

@torch.no_grad()
def predict(net, idx):
    net.eval(); out = []
    for xb, _ in DataLoader(Windows(idx, False), batch_size=256, num_workers=2):
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            out.append(torch.softmax(net(xb.to(DEVICE)).float(), 1).cpu().numpy())
    return np.concatenate(out)

tr = np.flatnonzero(win.split == "train"); va = np.flatnonzero(win.split == "val"); te = np.flatnonzero(win.split == "test")
counts = np.bincount(y[tr], minlength=4)
weights = torch.tensor((counts.sum() / (4 * counts)) ** 0.5, dtype=torch.float32)
print("train windows per class:", dict(zip(CLASSES, counts)), "\nloss weights:", weights.numpy().round(2))

In [ ]:
net = build_model(pretrained=True).to(DEVICE)
loader = DataLoader(Windows(tr, True), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(loader))
loss_fn = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

best, best_state, history = -1, None, []
for epoch in range(1, EPOCHS + 1):
    net.train(); t0 = time.time(); total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = loss_fn(net(xb), yb)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    m = icbhi_score(y[va], predict(net, va).argmax(1))
    history.append({"epoch": epoch, "loss": total / len(loader), **m})
    flag = ""
    if m["icbhi_score"] > best:
        best, best_state, flag = m["icbhi_score"], {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}, "  <- best"
    print(f"epoch {epoch:2d}  loss {total / len(loader):.3f}  val Sp {m['specificity']:.2f}  Se {m['sensitivity']:.2f}  score {m['icbhi_score']:.3f}  ({time.time() - t0:.0f} s){flag}")

net.load_state_dict(best_state)
pd.DataFrame(history).set_index("epoch")[["loss", "icbhi_score"]].plot(subplots=True, figsize=(10, 4)); plt.show()

## 5. Test on unseen patients

These patients were never used for training or for picking the model. **These are the numbers we report.** Report all of them, per class, and never claim clinical accuracy.

In [ ]:
p_test = predict(net, te)
pred = p_test.argmax(1)
test_metrics = icbhi_score(y[te], pred)
print(json.dumps(test_metrics, indent=2))
print(classification_report(y[te], pred, target_names=CLASSES, digits=3))
report = classification_report(y[te], pred, target_names=CLASSES, output_dict=True)

cm = confusion_matrix(y[te], pred, labels=range(4))
plt.figure(figsize=(4.5, 4)); plt.imshow(cm, cmap="Blues")
plt.xticks(range(4), CLASSES, rotation=30); plt.yticks(range(4), CLASSES)
for i in range(4):
    for j in range(4):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("test windows"); plt.show()

## 6. Export for the server (D5)

The server runs the model with ONNX Runtime, so it needs no PyTorch. We export, then check that the ONNX model gives the same answers as the PyTorch one.

In [ ]:
metrics = {"test_windows": test_metrics,
           "per_class": {c: {k: round(v, 3) for k, v in report[c].items()} for c in CLASSES},
           "split": {s: len(p) for s, p in sets.items()}}
export(net, EXPORT / "resp_cnn.onnx", MODEL_VERSION, metrics)

from inference import Analyzer
analyzer = Analyzer(EXPORT / "resp_cnn.onnx")
sample = te[:64]
diff = np.abs(analyzer.window_probs(X[sample].astype(np.float32)) - predict(net, sample)).max()
print(f"exported {MODEL_VERSION}; max difference ONNX vs PyTorch: {diff:.5f}")
assert diff < 1e-2

### Whole recordings, the way the server sees them

The app sends a 15 s recording per chest point, not single windows. Here we take 15 s from each test recording, pass it through the exact server path (`inference.Analyzer`), and compare with the true label of that stretch.

In [ ]:
CLIP_S = 15.0

def clips_of(row, hop_s=5.0):
    """15 s crops of one recording, at the device's 8 kHz/16-bit, with their true label."""
    audio, sr = sf.read(row.path, dtype="float32", always_2d=True)
    dev = icbhi.as_device(audio.mean(axis=1), sr)
    cycles = icbhi.read_cycles(row.path.replace(".wav", ".txt"))
    n = int(CLIP_S * pp.DEVICE_SR)
    last = max(0, len(dev) - n)
    for start in range(0, last + 1, int(hop_s * pp.DEVICE_SR)):
        label = icbhi.window_label(cycles, start / pp.DEVICE_SR, window_s=CLIP_S)
        if label is not None:
            yield start / pp.DEVICE_SR, dev[start:start + n], label

rows = []
for r in df[df.split == "test"].itertuples():
    for start, clip, label in clips_of(r):
        out = analyzer.analyze(clip, pp.DEVICE_SR)
        rows.append({"stem": r.stem, "point": r.point, "start": start, "true": CLASSES[label],
                     "pred": out["finding"], "confidence": out["confidence"], "audio": clip})
clips = pd.DataFrame(rows)
first = clips.groupby("stem").head(1)
print(f"{len(first)} test recordings, first 15 s each:")
print(classification_report(first.true, first.pred, labels=CLASSES, digits=3, zero_division=0))
metrics["test_recordings"] = icbhi_score(first.true.map(CLASSES.index), first.pred.map(CLASSES.index))
export(net, EXPORT / "resp_cnn.onnx", MODEL_VERSION, metrics)   # re-save with the recording-level scores
print(json.dumps(metrics["test_recordings"], indent=2))

## 7. Pick the demo clips (D6)

The judges are healthy, so a live recording will say "normal". Demo mode shows real abnormal sounds instead. For every case × chest point (4 × 7 = 28) we pick a 15 s test-set clip that the model gets **right**, with the highest confidence. Clips from the test set keep the demo honest: the model has never heard these patients.

If ICBHI has no correct clip for some point, we reuse the best clip of the same case from another point, and the manifest records that.

In [ ]:
from scipy.io import wavfile

DEMO = EXPORT / "demo_clips"; DEMO.mkdir(exist_ok=True)
ok = clips[clips.true == clips.pred]
manifest, sources = {}, []
for case in CLASSES:
    pool = ok[ok.true == case].sort_values("confidence", ascending=False)
    if pool.empty:
        print(f"WARNING: no correctly classified {case} clip in the test set; demo mode can't show {case}.")
        continue
    for point in POINTS:
        here = pool[pool.point == point]
        pick = (here if len(here) else pool).iloc[0]
        name = f"{case}_{point}.wav"
        wavfile.write(DEMO / name, pp.DEVICE_SR, pick.audio)
        manifest[f"{case}/{point}"] = name
        sources.append({"case": case, "point": point, "from_stem": pick.stem, "from_point": pick.point,
                        "start_s": pick.start, "confidence": pick.confidence, "same_point": len(here) > 0})
(DEMO / "manifest.json").write_text(json.dumps(manifest, indent=2))
src = pd.DataFrame(sources); src.to_csv(DEMO / "sources.csv", index=False)
print(f"{len(manifest)} demo clips written")
src.pivot(index="point", columns="case", values="confidence").reindex(POINTS)

## 8. Download

In [ ]:
(EXPORT / "results.json").write_text(json.dumps({"model_version": MODEL_VERSION, **metrics}, indent=2))
shutil.make_archive("/kaggle/working/respirai_export", "zip", EXPORT)
print("Done. Download respirai_export.zip from the Output section (right panel, or the notebook's Output tab).")
print(sorted(p.relative_to(EXPORT).as_posix() for p in EXPORT.rglob("*") if p.is_file())[:12], "...")

### Where the files go in the repo

| From the zip | Put it in | Commit it? |
|---|---|---|
| `resp_cnn.onnx`, `resp_cnn.json` | `ai/weights/` | no (too big; share the zip with the team instead) |
| `demo_clips/` (28 WAVs + `manifest.json` + `sources.csv`) | `ai/demo_clips/` | **yes**: the demo needs them, and they are small |
| `results.json`, `split.csv` | `ai/results/` | yes: these are the numbers we report |

Then start the server from the `ai/` folder with `uvicorn server:app --host 0.0.0.0 --port 8000` and open `http://localhost:8000/health`. It should show the new `model_version` and `"dummy": false`.